# Create, update, ACL, and delete an epic

`/admin/epics` stands in for engineering. The connector polls `updated_since`
about once a minute and writes `raw/engineering/epic/{id}.json`. A delete is
not on that listing. The reconcile sweep reads `/ids` and removes the snapshot.

An ACL change does not move the version. The snapshot is still rewritten
because the ACL changed.

Requires `make up`.


In [1]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "source_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import source_demo
engineering = source_demo.engineering


In [2]:
EPIC_ID = "epic-demo-notebook"
KEY = engineering.key(EPIC_ID, "json")
PAYLOAD = {
    "epic_id": EPIC_ID,
    "title": "Notebook epic",
    "status": "planned",
    "product_key": "notifications",
    "team": "platform",
    "native_acl": ["project_member"],
}
engineering.delete_if_present("epics", EPIC_ID)
created = engineering.admin("POST", "/admin/epics", PAYLOAD)
print(created["epic_id"], "version", created["version"])
engineering.expect_status("POST", "/admin/epics", 409, PAYLOAD)
engineering.expect_status(
    "POST", "/admin/epics", 422, {**PAYLOAD, "epic_id": "nope", "status": "shipped"}
)


epic-demo-notebook version 1


In [3]:
def written():
    body = source_demo.object_body(KEY) if source_demo.snapshot(KEY) else b""
    return b"Notebook epic" in body

source_demo.wait_until(written, source_demo.WRITE_WAIT, "epic snapshot")
print("snapshot", KEY)


snapshot raw/engineering/epic/epic-demo-notebook.json


In [4]:
edited = engineering.admin("PATCH", f"/admin/epics/{EPIC_ID}", {"status": "in_progress"})
print("version", edited["version"], edited["status"])

def changed():
    return b"in_progress" in source_demo.object_body(KEY)

source_demo.wait_until(changed, source_demo.WRITE_WAIT, "epic status")


version 2 in_progress


True

In [5]:
before = engineering.admin("GET", f"/epics/{EPIC_ID}")["version"]
changed = engineering.admin(
    "PUT", f"/admin/epics/{EPIC_ID}/acl", {"native_acl": ["eng_leadership"]}
)
print("version stayed" if changed["version"] == before else "version moved", changed["version"])

def acl_stored():
    head = source_demo.snapshot(KEY)
    return head is not None and "eng_leadership" in head["Metadata"].get(source_demo.META_ACL, "")

source_demo.wait_until(acl_stored, source_demo.WRITE_WAIT, "epic acl")


version stayed 2


True

In [6]:
engineering.admin("DELETE", f"/admin/epics/{EPIC_ID}")
source_demo.wait_until(
    lambda: source_demo.snapshot(KEY) is None, source_demo.GONE_WAIT, "epic deleted"
)
print("deleted", EPIC_ID)


deleted epic-demo-notebook
